> Historical notebook: the original workflow is preserved, but outputs are cleared to avoid stale results and personal paths. See reports/refinement_report.md and scripts/run_coursework.py for the corrected chronological comparison.


# NLP-RL Trading Platform Result Overview Dashboard

This notebook is a read-only consolidated review surface. It scans existing outputs, summarizes metrics, displays available charts, and writes lightweight overview exports. It does not scrape data or train DQN models.

# Coverage-Controlled Experimental Design

This notebook is a cached-output review dashboard. It does not scrape or train.

- Long historical market data is used to learn market-based trading behavior.
- For each stock, the recent 80% of news observations defines the high-information-density evaluation window.
- NLP value is judged mainly inside that dense window, where text information is available often enough to be meaningful.
- Cross-stock robustness uses the common overlap of each stock's dense window.
- No-news days are not neutral sentiment; they are marked with `news_available=0` and `sentiment_missing_flag=1`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORTS = PROJECT / "reports"
TABLES = REPORTS / "tables"
SYSTEM = PROJECT / "outputs" / "system"
STOCKS = PROJECT / "outputs" / "stocks"

def read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 4:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, **kwargs)
    except Exception as exc:
        display(Markdown(f"Warning: could not read `{path}`: {exc}"))
        return pd.DataFrame()

density = read_csv(TABLES / "information_density_split.csv", dtype={"symbol": str})
windows = read_csv(TABLES / "experiment_window_summary.csv", dtype={"symbol": str})
cross_hd = read_csv(SYSTEM / "cross_stock_high_density_summary.csv", dtype={"symbol": str})
missing = read_csv(REPORTS / "result_overview_missing_outputs.csv", dtype={"symbol": str})

display(Markdown("## Information Density Summary"))
if density.empty:
    display(Markdown("Information-density outputs are missing. Run `scripts/generate_information_density_outputs.py`."))
else:
    display(density)

display(Markdown("## Experiment Window Summary"))
display(windows if not windows.empty else pd.DataFrame({"warning": ["experiment_window_summary.csv missing"]}))

display(Markdown("## Cross-Stock High-Density Summary"))
if cross_hd.empty:
    display(Markdown("Cross-stock high-density summary not found or empty."))
else:
    display(cross_hd)
    cols = [c for c in ["nlp_basic_final_equity_effect", "nlp_enhanced_final_equity_effect", "nlp_basic_sharpe_effect"] if c in cross_hd.columns]
    if cols:
        ax = cross_hd.set_index("symbol")[cols].apply(pd.to_numeric, errors="coerce").plot(kind="bar", figsize=(11, 4), title="High-density NLP effects")
        ax.axhline(0, color="black", linewidth=0.8)
        plt.tight_layout()
        plt.show()

display(Markdown("## Reliability and Missing Outputs"))
display(missing if not missing.empty else pd.DataFrame({"warning": ["missing-output index not generated"]}))


## Model Behavior & Trading Visualization Gallery

These charts are loaded from cached output files. They use the selected evaluation window where high-density files are available, and they do not rerun scraping or DQN training.


In [ ]:
from IPython.display import Image, display, Markdown
visual_index = read_csv(TABLES / "trading_visualization_index.csv", dtype={"symbol": str})
display(Markdown("### Visualization File Status"))
display(visual_index if not visual_index.empty else pd.DataFrame({"warning": ["trading_visualization_index.csv missing; run scripts/generate_trading_visualizations.py"]}))

preferred_symbols = []
if not density.empty and "symbol" in density.columns:
    preferred_symbols = density["symbol"].astype(str).str.zfill(6).head(3).tolist()
elif STOCKS.exists():
    preferred_symbols = sorted(p.name for p in STOCKS.iterdir() if p.is_dir() and p.name.isdigit())[:3]

visual_files = [
    ("price_with_trading_actions.png", "Close price with Buy/Sell/Hold markers"),
    ("portfolio_value_comparison.png", "Portfolio value comparison across strategies"),
    ("action_distribution.png", "Buy/Sell/Hold action distribution"),
    ("trade_outcome_win_rate.png", "Trade outcome / win-rate visualization"),
    ("drawdown_curve_comparison.png", "Drawdown comparison"),
    ("sentiment_action_overlay.png", "Sentiment and action overlay"),
    ("prediction_vs_actual_direction.png", "Prediction vs actual direction, if available"),
]
for sym in preferred_symbols:
    display(Markdown(f"### Stock {sym}"))
    report_dir = STOCKS / sym / "reports"
    for filename, caption in visual_files:
        path = report_dir / filename
        if path.exists() and path.stat().st_size > 4:
            display(Markdown(f"**{caption}**"))
            display(Image(filename=str(path)))
        else:
            display(Markdown(f"Warning: `{path.relative_to(PROJECT)}` is missing."))

display(Markdown("### Cross-Stock Visual Summary"))
cross_visual = SYSTEM / "figures" / "cross_stock_visual_summary.png"
cross_visual_table = SYSTEM / "cross_stock_visual_summary.csv"
if cross_visual_table.exists():
    display(read_csv(cross_visual_table, dtype={"symbol": str}))
if cross_visual.exists() and cross_visual.stat().st_size > 4:
    display(Image(filename=str(cross_visual)))
else:
    display(Markdown("Warning: cross-stock visual summary image is missing."))


## Final Interpretation Template

NLP sentiment enters the project in two places: first as text-to-signal conversion from raw news into daily sentiment, and second as lagged state input for the DQN policy. The basic NLP state keeps the guideline-required `sentiment_score`; the enhanced state adds coverage-aware text features such as `news_available`, `news_count`, and rolling sentiment.

The result should be read cautiously. NLP improves DQN only when the high-density window has enough text coverage and when the improvement is larger than seed-level noise. If the common high-density overlap is too short, cross-stock evidence is marked `NOT_RELIABLE` rather than overclaimed.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image, SVG

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS = PROJECT_ROOT / 'outputs'
STOCK_ROOT = OUTPUTS / 'stocks'
SYSTEM = OUTPUTS / 'system'
REPORTS = PROJECT_ROOT / 'reports'
TABLES = REPORTS / 'tables'

def read_csv(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 4:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f'Could not read {path}: {exc}')
        return pd.DataFrame()

def latest_file(directory, pattern):
    directory = Path(directory)
    if not directory.exists():
        return None
    files = sorted([p for p in directory.glob(pattern) if p.is_file() and p.stat().st_size > 4], key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None

def detected_stocks():
    if not STOCK_ROOT.exists():
        return []
    return sorted([p.name for p in STOCK_ROOT.iterdir() if p.is_dir() and p.name.isdigit()])

def stock_paths(symbol):
    root = STOCK_ROOT / symbol
    data = latest_file(root / 'data', '*_finance_text_*.csv')
    if data and data.name.endswith('_master.csv'):
        non_master = [p for p in sorted((root / 'data').glob('*_finance_text_*.csv')) if not p.name.endswith('_master.csv')]
        data = non_master[-1] if non_master else data
    return {
        'data': data,
        'nlp_eval': latest_file(root / 'reports', '*_nlp_evaluation.csv'),
        'daily_sentiment': latest_file(root / 'reports', '*_daily_sentiment.csv'),
        'ablation': root / 'results' / 'ablation_metrics.csv',
        'curves': root / 'results' / 'portfolio_curves.csv',
        'figures': sorted((root / 'reports').glob('*.svg')) if (root / 'reports').exists() else [],
        'diagnostics': latest_file(root / 'reports', '*_diagnostics.csv'),
    }

def classify_nlp_effect(metrics):
    if metrics.empty or 'experiment' not in metrics.columns or 'final_equity' not in metrics.columns:
        return 'Inconclusive'
    rows = metrics.set_index('experiment')
    if not {'dqn_with_nlp', 'dqn_without_nlp'}.issubset(rows.index):
        return 'Inconclusive'
    eq_delta = pd.to_numeric(pd.Series([rows.loc['dqn_with_nlp', 'final_equity']]), errors='coerce').iloc[0] - pd.to_numeric(pd.Series([rows.loc['dqn_without_nlp', 'final_equity']]), errors='coerce').iloc[0]
    sh_delta = 0
    if 'sharpe_ratio' in rows.columns:
        sh_delta = pd.to_numeric(pd.Series([rows.loc['dqn_with_nlp', 'sharpe_ratio']]), errors='coerce').iloc[0] - pd.to_numeric(pd.Series([rows.loc['dqn_without_nlp', 'sharpe_ratio']]), errors='coerce').iloc[0]
    if pd.isna(eq_delta):
        return 'Inconclusive'
    if eq_delta > 0 and sh_delta > 0:
        return 'NLP improves'
    if eq_delta < 0 and sh_delta < 0:
        return 'NLP hurts'
    return 'Mixed'

def show_file(path):
    if not path or not Path(path).exists():
        display(Markdown('Missing'))
        return
    path = Path(path)
    if path.suffix.lower() == '.svg':
        display(SVG(filename=str(path)))
    elif path.suffix.lower() == '.png':
        display(Image(filename=str(path)))
    elif path.suffix.lower() == '.csv':
        display(read_csv(path).head(30))
    elif path.suffix.lower() == '.md':
        display(Markdown(path.read_text(encoding='utf-8')))

stocks = detected_stocks()
display(Markdown(f'Generated: `{datetime.now().isoformat(timespec="seconds")}`'))
display(Markdown(f'Detected stocks: `{", ".join(stocks) if stocks else "none"}`'))

## 1. Project Result Overview

In [ ]:
display(Markdown('**Research question:** Do NLP signals improve DQN trading performance?'))
overview_rows = []
for symbol in stocks:
    paths = stock_paths(symbol)
    data = read_csv(paths['data']) if paths['data'] else pd.DataFrame()
    metrics = read_csv(paths['ablation'])
    sentiment = read_csv(paths['daily_sentiment']) if paths['daily_sentiment'] else pd.DataFrame()
    dates = pd.to_datetime(data.get('date', pd.Series(dtype=str)), errors='coerce') if not data.empty else pd.Series(dtype='datetime64[ns]')
    nlp_eval = read_csv(paths['nlp_eval']) if paths['nlp_eval'] else pd.DataFrame()
    finbert = ', '.join(sorted(set(nlp_eval.get('finbert_status', pd.Series(dtype=str)).dropna().astype(str)))) if not nlp_eval.empty else 'missing'
    coverage = float((pd.to_numeric(sentiment.get('news_count', pd.Series(dtype=float)), errors='coerce').fillna(0) > 0).mean()) if not sentiment.empty and 'news_count' in sentiment.columns else None
    best = metrics.sort_values('final_equity', ascending=False)['experiment'].iloc[0] if not metrics.empty and {'experiment','final_equity'}.issubset(metrics.columns) else 'missing'
    overview_rows.append({
        'symbol': symbol,
        'date_start': dates.min().date() if dates.notna().any() else '',
        'date_end': dates.max().date() if dates.notna().any() else '',
        'data_rows': len(data),
        'sentiment_coverage': coverage,
        'finbert_status': finbert,
        'best_strategy': best,
        'nlp_effect': classify_nlp_effect(metrics),
    })
overview = pd.DataFrame(overview_rows)
display(overview)

## 2. Output Discovery

In [ ]:
discovery = []
for symbol in stocks:
    paths = stock_paths(symbol)
    discovery.append({
        'symbol': symbol,
        'integrated_csv': bool(paths['data']),
        'nlp_eval': bool(paths['nlp_eval']),
        'ablation_metrics': paths['ablation'].exists(),
        'portfolio_curves': paths['curves'].exists(),
        'figures': len(paths['figures']),
        'diagnostics': bool(paths['diagnostics']),
    })
discovery_df = pd.DataFrame(discovery)
display(discovery_df)
display(Markdown(f'Cross-stock summary exists: `{(SYSTEM / "cross_stock_summary.csv").exists()}`'))
display(Markdown(f'Feasibility audit exists: `{(SYSTEM / "reports" / "feasibility_audit.csv").exists()}`'))

## 3. Single-Stock Visual Gallery

In [ ]:
wanted_figures = ['close_price', 'sentiment_trend', 'daily_news_count', 'portfolio_curves', 'final_portfolio_values', 'drawdown_curves', 'risk_return_scatter', 'trading_activity', 'signal_return_correlations']
gallery_rows = []
for symbol in stocks:
    display(Markdown(f'### {symbol}'))
    paths = stock_paths(symbol)
    figures = {fig.stem: fig for fig in paths['figures']}
    for wanted in wanted_figures:
        matches = [fig for fig in paths['figures'] if wanted in fig.stem]
        gallery_rows.append({'symbol': symbol, 'figure': wanted, 'status': 'Complete' if matches else 'Missing'})
        if matches:
            display(Markdown(f'**{wanted}**'))
            show_file(matches[-1])
        else:
            display(Markdown(f'- Missing `{wanted}`'))
gallery_status = pd.DataFrame(gallery_rows)
display(gallery_status)

## 4. NLP Evaluation Summary

In [ ]:
nlp_rows = []
for symbol in stocks:
    path = stock_paths(symbol)['nlp_eval']
    frame = read_csv(path) if path else pd.DataFrame()
    if not frame.empty:
        frame.insert(0, 'symbol', symbol)
        frame['evaluation_type'] = frame.get('status', '').astype(str).where(~frame.get('status', '').astype(str).str.contains('pseudo', case=False, na=False), 'pseudo_label_eval')
        nlp_rows.append(frame)
combined_nlp = pd.concat(nlp_rows, ignore_index=True) if nlp_rows else pd.DataFrame()
gold = read_csv(TABLES / 'nlp_gold_label_evaluation.csv')
display(Markdown('**Warning:** pseudo-label F1 is not gold-label F1.'))
display(combined_nlp if not combined_nlp.empty else 'No stock NLP evaluation files found.')
display(Markdown('### Gold-label / final NLP comparison'))
display(gold if not gold.empty else 'Gold-label evaluation is missing.')

## 5. Ablation Summary

In [ ]:
ablation_rows = []
for symbol in stocks:
    metrics = read_csv(stock_paths(symbol)['ablation'])
    if not metrics.empty:
        metrics.insert(0, 'symbol', symbol)
        metrics['conclusion'] = classify_nlp_effect(metrics)
        ablation_rows.append(metrics)
ablation = pd.concat(ablation_rows, ignore_index=True) if ablation_rows else pd.DataFrame()
cols = [c for c in ['symbol','experiment','final_equity','cumulative_return','sharpe_ratio','max_drawdown','sortino_ratio','calmar_ratio','number_of_trades','exposure_ratio','conclusion'] if c in ablation.columns]
display(ablation[cols] if cols else 'No ablation metrics found.')
if not ablation.empty and {'symbol','experiment','final_equity'}.issubset(ablation.columns):
    pivot = ablation.pivot_table(index='symbol', columns='experiment', values='final_equity')
    pivot.plot(kind='bar', figsize=(10,4), title='Final Equity by Stock and Strategy')
    plt.tight_layout(); plt.show()

## 6. Cross-Stock Summary

In [ ]:
cross = read_csv(SYSTEM / 'cross_stock_summary.csv')
if cross.empty:
    display(Markdown('Cross-stock summary not found. Run optional cross-stock robustness analysis first.'))
else:
    display(cross)
    for metric in ['final_equity', 'sharpe_ratio']:
        strategy_cols = [c for c in [f'buy_and_hold_{metric}', f'dqn_without_nlp_{metric}', f'dqn_with_nlp_{metric}'] if c in cross.columns]
        if strategy_cols:
            cross.set_index('symbol')[strategy_cols].plot(kind='bar', figsize=(10,4), title=f'{metric} by stock and strategy')
            plt.tight_layout(); plt.show()
    if {'sentiment_coverage_ratio','nlp_final_equity_effect','symbol'}.issubset(cross.columns):
        ax = cross.plot.scatter(x='sentiment_coverage_ratio', y='nlp_final_equity_effect', figsize=(6,4), title='Sentiment coverage vs NLP equity effect')
        for _, row in cross.iterrows():
            ax.annotate(row['symbol'], (row['sentiment_coverage_ratio'], row['nlp_final_equity_effect']))
        plt.tight_layout(); plt.show()

## 7. Feasibility Audit Summary

In [ ]:
for name in ['feasibility_audit.csv', 'cross_stock_feasibility_summary.csv', 'missing_file_report.csv', 'data_quality_diagnostics.csv']:
    path = SYSTEM / 'reports' / name
    display(Markdown(f'### {name}'))
    frame = read_csv(path)
    display(frame if not frame.empty else f'{name} missing or empty')
md = SYSTEM / 'reports' / 'feasibility_audit.md'
if md.exists():
    display(Markdown(md.read_text(encoding='utf-8')))

## 8. Final Report Readiness Checklist

In [ ]:
checks = [
    REPORTS / 'final_report.md',
    REPORTS / 'final_report_figure_index.csv',
    REPORTS / 'final_report_table_index.csv',
    REPORTS / 'presentation_outline.md',
    REPORTS / 'live_demo_script.md',
    REPORTS / 'github_submission_checklist.md',
    PROJECT_ROOT / 'README.md',
    PROJECT_ROOT / 'requirements.txt',
    SYSTEM / 'cross_stock_summary.csv',
    TABLES / 'nlp_gold_label_evaluation.csv',
    REPORTS / 'sqlite_demo.md',
]
readiness = pd.DataFrame([{'file': str(p.relative_to(PROJECT_ROOT)), 'status': 'Complete' if p.exists() and p.stat().st_size > 4 else 'Missing'} for p in checks])
display(readiness)

## 8.5 Model Reliability and Optimization Diagnostics


In [ ]:
stability = read_csv(TABLES / 'model_stability_summary.csv')
actions = read_csv(TABLES / 'action_behavior_diagnostics.csv')
enhanced_ablation = read_csv(TABLES / 'enhanced_nlp_ablation_metrics.csv')
enhanced_signals = read_csv(TABLES / 'signal_diagnostics_enhanced.csv')
display(Markdown('### DQN stability across seeds'))
display(stability if not stability.empty else 'model_stability_summary.csv missing')
display(Markdown('### Action behavior: conservative vs overtrading'))
display(actions if not actions.empty else 'action_behavior_diagnostics.csv missing')
display(Markdown('### Enhanced NLP ablation status'))
display(enhanced_ablation if not enhanced_ablation.empty else 'enhanced_nlp_ablation_metrics.csv missing')
display(Markdown('### Enhanced signal predictive diagnostics'))
display(enhanced_signals if not enhanced_signals.empty else 'signal_diagnostics_enhanced.csv missing')
reliability_notes = []
if not stability.empty and stability.get('high_variance_warning', pd.Series(dtype=bool)).fillna(False).any():
    reliability_notes.append('Some DQN rows have high seed-level variance; improvements smaller than std should be treated as inconclusive.')
if not actions.empty and (actions.get('warning', pd.Series(dtype=str)).astype(str) != 'ok').any():
    reliability_notes.append('Some action diagnostics indicate possible over-conservatism or overtrading.')
if not enhanced_ablation.empty and 'not_run_by_default' in set(enhanced_ablation.get('enhanced_nlp_status', pd.Series(dtype=str)).astype(str)):
    reliability_notes.append('Enhanced NLP ablation has not been trained by default; do not claim enhanced NLP improvement until explicitly run.')
if not combined_nlp.empty and combined_nlp.astype(str).apply(lambda col: col.str.contains('skipped|pseudo_label_eval', case=False, na=False)).any().any():
    reliability_notes.append('FinBERT skipped and/or pseudo-label-only NLP evaluation exists; NLP evidence should be interpreted with caution.')
display(Markdown('\n'.join(f'- {note}' for note in reliability_notes) if reliability_notes else 'No major model reliability warnings detected from cached outputs.'))


## 9. Auto-generated Interpretation and Export

In [ ]:
effect_counts = overview['nlp_effect'].value_counts().to_dict() if 'overview' in globals() and not overview.empty else {}
cross_status = ', '.join(sorted(set(cross.get('comparability_status', pd.Series(dtype=str)).dropna().astype(str)))) if 'cross' in globals() and not cross.empty else 'missing'
missing = readiness[readiness['status'] != 'Complete'] if 'readiness' in globals() else pd.DataFrame()
missing_text = missing.to_string(index=False) if not missing.empty else 'No tracked readiness outputs are missing.'
summary_md = f"""
# Result Overview Summary

- Generated: `{datetime.now().isoformat(timespec='seconds')}`
- Detected stocks: `{len(stocks)}`
- NLP effect counts: `{effect_counts}`
- Cross-stock comparability: `{cross_status}`

## Overall Result

NLP does not consistently improve DQN performance across all stocks. It can help some stocks and hurt others depending on sentiment coverage, signal quality, market regime, and conservative time alignment.

## Data Quality

The project has cached local outputs and diagnostics. Cross-stock comparison should be interpreted using the common-window comparability status.

## Main Limitations

- FinBERT must be checked as `ok` before claiming it ran.
- Pseudo-label F1 is not gold-label F1.
- Walk-forward evidence is labelled honestly unless true rolling retraining is run.
- News density can be uneven across time and stocks.

## Remaining Missing Outputs

{missing_text}
"""
display(Markdown(summary_md))
REPORTS.mkdir(parents=True, exist_ok=True)
Path(REPORTS / 'result_overview_summary.md').write_text(summary_md, encoding='utf-8')
overview.to_csv(REPORTS / 'result_overview_tables.csv', index=False, encoding='utf-8-sig')
missing.to_csv(REPORTS / 'result_overview_missing_outputs.csv', index=False, encoding='utf-8-sig')
display(Markdown('Exports written: `reports/result_overview_summary.md`, `reports/result_overview_tables.csv`, `reports/result_overview_missing_outputs.csv`.'))

## Peer-Sector NLP Transfer Official Results

This is the latest official experiment. The target stock is held out from NLP training. Sector-peer and marketwide-peer corpora score the target stock high-density news. Legacy stock-level NLP is deprecated and should not be used as the main conclusion.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
peer_files = {
    "sector_mapping": ROOT / "reports" / "tables" / "stock_sector_mapping.csv",
    "corpus_summary": ROOT / "reports" / "tables" / "peer_nlp_corpus_summary.csv",
    "effect_summary": ROOT / "reports" / "tables" / "peer_nlp_effect_summary.csv",
    "cross_stock_summary": ROOT / "outputs" / "system" / "peer_nlp_cross_stock_summary.csv",
    "integrity_check": ROOT / "reports" / "tables" / "peer_nlp_integrity_check.csv",
}
status_rows = []
for name, path in peer_files.items():
    status_rows.append({"artifact": name, "exists": path.exists(), "path": str(path.relative_to(ROOT))})
display(pd.DataFrame(status_rows))
for name, path in peer_files.items():
    if path.exists() and path.stat().st_size > 4:
        display(Markdown(f"### {name}"))
        display(pd.read_csv(path).head(20))
    else:
        display(Markdown(f"**Missing:** `{path.relative_to(ROOT)}`. Run the official peer NLP workflow before using this notebook section as evidence."))


### Peer NLP Interpretation Checklist

- Target stock excluded from sector and marketwide NLP corpora.
- Sector corpus has at least four peer stocks or is marked insufficient.
- Peer training news thresholds are met, or the result is marked not reliable.
- Target high-density test window has at least 30 trading days and sufficient news coverage.
- DQN portfolios are not flat and test trades are positive.
- Legacy stock-level NLP outputs are not used as official conclusions.
